In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [2]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.checkpoint.sqlite import SqliteSaver

from typing_extensions import TypedDict

import sqlite3

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [4]:
class Agent:

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
    
    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        print(f"Mensajes enviados al modelo:", messages)
        message = self.model.invoke(messages)
        return {'messages': [message]}
    
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0
    
    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Llamando la Herramienta: {t['name']} con los siguientes argumentos: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Retornando a la LLM tras la acción!")
        return {'messages': results}

In [11]:
# current_tavily_api_key = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY no encontrada. Asegúrate de que esté en tu .env y de que python-dotenv esté instalado.")

tool = TavilySearch(max_results=3, tavily_api_key=TAVILY_API_KEY)

prompt_system = """
Usa la fecha de hoy como referencia para responder preguntas sobre el clima, eventos, noticias y otros temas de actualidad.
Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.
Tienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).
Busca información únicamente cuando tengas certeza de qué buscar.
Si necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.
Cuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.
"""

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

abot = Agent(model, [tool], system=prompt_system, checkpointer=memory)

In [14]:
messages = [HumanMessage(content="¿Cómo está el clima en Lima, Peru?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pregunta 1: Clima en Lima ---
Mensajes enviados al modelo: [SystemMessage(content='\nUsa la fecha de hoy como referencia para responder preguntas sobre el clima, eventos, noticias y otros temas de actualidad.\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?', additional_kwargs={}, response_metadata={}), HumanMessag

In [15]:
messages = [HumanMessage(content="Y el clima en Medellin?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 2: Clima en Medellin ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pregunta 2: Clima en Medellin ---
Mensajes enviados al modelo: [SystemMessage(content='\nUsa la fecha de hoy como referencia para responder preguntas sobre el clima, eventos, noticias y otros temas de actualidad.\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?', additional_kwargs={}, response_metadata={}), HumanMe

In [18]:
messages = [HumanMessage(content="cual de las ciudades de la memoria está más caliente?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 3: Comparacion ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pregunta 3: Comparacion ---
Mensajes enviados al modelo: [SystemMessage(content='\nUsa la fecha de hoy como referencia para responder preguntas sobre el clima, eventos, noticias y otros temas de actualidad.\nEres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nSi necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.\nCuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?', additional_kwargs={}, response_metadata={}), HumanMessage(

In [27]:
config = {"configurable": {"thread_id": "2"}}
estado = abot.graph.get_state(config)
# print(estado.values)

for mensaje in estado.values["messages"]:
    print(type(mensaje).__name__, ":", mensaje.content)

HumanMessage : ¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?
HumanMessage : ¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?
AIMessage : No puedo darte información sobre el clima en Lima, Brasil, ya que esa ciudad no existe. ¿Quizás te refieres a Lima, Perú?
HumanMessage : ¿Cómo está el clima en Lima?
AIMessage : Para poder ofrecerte información precisa sobre el estado del tiempo, necesito que me confirmes si te refieres a Lima, Perú, o a alguna otra ciudad con ese nombre.
HumanMessage : Y el clima en Medellin?
AIMessage : 
ToolMessage : {'query': 'clima en Medellín', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://es.numbeo.com/clima/ciudad/Medellin', 'title': 'Clima en Medellín', 'content': 'Title: Clima en Medellín\n# Clima en Medellín. | Índice de Clima: | 99,76 |. ## Clima por Mes en Medellín. | Mes | Puntuación |del Clima Resumen de Condiciones Climáticas |. | Febrero | 100,00 | Posibilidad de humeda

In [28]:
cursor = conn.cursor()

cursor.execute("""
    SELECT DISTINCT thread_id
    FROM checkpoints
""")

threads = cursor.fetchall()

print("Threads:", threads)
print("Cantidad:", len(threads))

Threads: [('2',)]
Cantidad: 1
